## 1: Setup and Dependencies

**What you'll learn:** Environment setup, AWS configuration, and creating Memory and Knowledge Base resources.

**Why it matters:** Proper foundation ensures smooth deployment. Memory enables personalized conversations, while Knowledge Base provides domain-specific information retrieval.

**Real-world value:** Memory strategies (preferences, semantic facts, summaries) let agents remember customer history across sessions—like a sales rep who recalls your preferences from last month.

**Analogy:** Setting up a new employee's workspace: computer (dependencies), ID badge (AWS credentials), filing cabinet (Memory), and reference library (Knowledge Base).

![Architecture](images/Architecture.png)

---

**Prerequisites:** Python 3.12+, AWS account with Bedrock permissions

### Step 1: Install Dependencies and Import Required Libraries

In [1]:
# Install required packages
! pip install -U -r requirements.txt -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
! python.exe -m pip install --upgrade pip

In [9]:
import os

# Set the AWS profile to match your shell environment
os.environ['AWS_PROFILE'] = 'workshop-profile'

In [10]:
# Import required libraries and verify AWS configuration
import json
import boto3
from strands import Agent
from strands.models import BedrockModel

session = boto3.Session()
sts = session.client('sts')
identity = sts.get_caller_identity()
account_id = identity['Account']
region = session.region_name or 'us-west-2'

print(f"✅ Account ID: {account_id}")
print(f"✅ Region: {region}")

✅ Account ID: 625579972148
✅ Region: us-west-2


### Step 2: Prepare Memory and Knowledge Base

This creates AgentCore Memory with three strategies: User Preferences, Semantic memory for conversation facts, and Summary memory for conversation context.

**Note:** This may take a few minutes. While waiting, try asking Kiro (chat on right) about AgentCore Memory concepts.

For eg: "What is Bedrock AgentCore Memory? Why is it needed? Explain to me with a real-world analogy."

In [11]:
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

MEMORY_NAME="ReturnRefundAssisantMemory"

memory_manager = MemoryManager(region_name=region)

# Create or get memory with strategies
memory = memory_manager.get_or_create_memory(
    name=MEMORY_NAME,
    description="Memory for returns and refunds assistant",
    strategies=[
                {
                    StrategyType.USER_PREFERENCE.value: {
                        "name": "CustomerPreferences",
                        "description": "Captures customer preferences and behavior",
                        "namespaces": ["returns/customer/{actorId}/preferences"],
                    }
                },
                {
                    StrategyType.SEMANTIC.value: {
                        "name": "CustomerSupportSemantic",
                        "description": "Stores facts from conversations",
                        "namespaces": ["returns/customer/{actorId}/semantic"],
                    }
                },
                {
                    StrategyType.SUMMARY.value: {
                        "name": "ConversationSummary",
                        "description": "Maintains conversation context and summaries",
                        "namespaces": ["returns/customer/{actorId}/{sessionId}/summary"],
                    }
                },
            ]
)

memory_id = memory["id"]

# Save memory configuration to JSON file
memory_config = {
    "memory_id": memory_id,
    "memory_name": MEMORY_NAME
}
with open('memory_config.json', 'w') as f:
    json.dump(memory_config, f, indent=2)

print(f"✅ Memory configuration saved to memory_config.json")

✅ MemoryManager initialized for region: us-west-2
Created memory: ReturnRefundAssisantMemory-3ar3Bl9YAh
Created memory ReturnRefundAssisantMemory-3ar3Bl9YAh, waiting for ACTIVE status...
Waiting for memory ReturnRefundAssisantMemory-3ar3Bl9YAh to return to ACTIVE state and strategies to reach terminal states...


[00:44:03]    ⏳ Memory: CREATING, Strategies: 0/3 active (10s elapsed)                             ]8;id=755106;file://c:\dev\sample-aiml-solution-labs\venv_refund\Lib\site-packages\bedrock_agentcore_starter_toolkit\operations\memory\manager.py\manager.py]8;;\:]8;id=75461;file://c:\dev\sample-aiml-solution-labs\venv_refund\Lib\site-packages\bedrock_agentcore_starter_toolkit\operations\memory\manager.py#1023\1023]8;;\

[00:44:13]    ⏳ Memory: CREATING, Strategies: 0/3 active (20s elapsed)                             ]8;id=513130;file://c:\dev\sample-aiml-solution-labs\venv_refund\Lib\site-packages\bedrock_agentcore_starter_toolkit\operations\memory\manager.py\manager.py]8;;\:]8;id=294749;file://c:\dev\sample-aiml-solution-labs\venv_refund\Lib\site-packages\bedrock_agentcore_starter_toolkit\operations\memory\manager.py#1023\1023]8;;\

[00:44:24]    ⏳ Memory: CREATING, Strategies: 0/3 active (30s elapsed)                             ]8;id=711624;file://c:\dev\sample-aiml-solution-labs\venv_refund\Lib\site-packages\bedrock_agentcore_starter_toolkit\operations\memory\manager.py\manager.py]8;;\:]8;id=685876;file://c:\dev\sample-aiml-solution-labs\venv_refund\Lib\site-packages\bedrock_agentcore_starter_toolkit\operations\memory\manager.py#1023\1023]8;;\

[00:44:34]    ⏳ Memory: CREATING, Strategies: 0/3 active (41s elapsed)                             ]8;id=163387;file://c:\dev\sample-aiml-solution-labs\venv_refund\Lib\site-packages\bedrock_agentcore_starter_toolkit\operations\memory\manager.py\manager.py]8;;\:]8;id=654794;file://c:\dev\sample-aiml-solution-labs\venv_refund\Lib\site-packages\bedrock_agentcore_starter_toolkit\operations\memory\manager.py#1023\1023]8;;\

...

              ✅ Memory is ACTIVE (took 175s)                                                       ]8;id=374939;file://c:\dev\sample-aiml-solution-labs\venv_refund\Lib\site-packages\bedrock_agentcore_starter_toolkit\operations\memory\manager.py\manager.py]8;;\:]8;id=955094;file://c:\dev\sample-aiml-solution-labs\venv_refund\Lib\site-packages\bedrock_agentcore_starter_toolkit\operations\memory\manager.py#1043\1043]8;;\

ObservabilityDeliveryManager initialized for region: us-west-2, account: 625579972148
Created log group: /aws/vendedlogs/bedrock-agentcore/memory/APPLICATION_LOGS/ReturnRefundAssisantMemory-3ar3Bl9YAh
✅ Logs delivery enabled for memory/ReturnRefundAssisantMemory-3ar3Bl9YAh
✅ Traces delivery enabled for memory/ReturnRefundAssisantMemory-3ar3Bl9YAh
Observability enabled for memory/ReturnRefundAssisantMemory-3ar3Bl9YAh - logs: True, traces: True


✅ Observability enabled for memory ReturnRefundAssisantMemory-3ar3Bl9YAh

Log group: /aws/vendedlogs/bedrock-agentcore/memory/APPLICATION_LOGS/ReturnRefundAssisantMemory-3ar3Bl9YAh

✅ Memory configuration saved to memory_config.json


### Step 3: Retrieve Knowledge Base ID

Get the Knowledge Base ID from the CloudFormation stack output. This Knowledge Base contains the return and refund policies.

In [12]:
# Retrieve the knowledge base ID from CloudFormation stack outputs
cfn_client = boto3.client('cloudformation', region_name=region)

try:
    response = cfn_client.describe_stacks(StackName='knowledgebase')
    outputs = response['Stacks'][0]['Outputs']
    
    kb_id = None
    for output in outputs:
        if output['OutputKey'] == 'KnowledgeBaseId':
            kb_id = output['OutputValue']
            break
    
    if kb_id:
        print(f"✅ Knowledge Base ID: {kb_id}")
    else:
        raise ValueError("KnowledgeBaseId output not found in stack")
        
except Exception as e:
    print(f"⚠️ Error retrieving from CloudFormation: {e}")
    kb_id = "<PLACE-YOUR-KB-ID>"

# Save knowledge base configuration to JSON file
kb_config = {
    "kb_id": kb_id
}
with open('kb_config.json', 'w') as f:
    json.dump(kb_config, f, indent=2)

print(f"✅ Knowledge Base configuration saved to kb_config.json")

✅ Knowledge Base ID: EK2IHAXS8Q
✅ Knowledge Base configuration saved to kb_config.json


### Summary

You've installed dependencies, configured AWS credentials, and created Memory and Knowledge Base resources.

### Next Steps

- **2: AgentCore Identity** - Configure secure authentication